In [1]:
!pip install fuzzywuzzy python-Levenshtein

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------- ----- 1.3/1.5 MB 8.4 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 3.3 MB/s  0:00:00

   ---------------------------------------- 0/3 [rapidfuzz]
   ---------------------------------------- 0/3 [rapidfuzz]
   ---------------------------------------- 0/3 [rapidfuzz]
   ---------------------------------------- 0/3 [rapidfuzz]
   ---------------------------------------- 0/3 [rapidfuzz]
   ---------------------------------------- 0/3 [rapidfuzz]
   ---------------------------------------- 0/3 [rapidfuzz]
   ---------------------------------------- 0/3 [rapidfuzz]
   ---------------------------------------- 0/3 [rapidfuzz]
   ---------------------------------------- 0/3 [rapidfuzz]
   ---------------------------------------- 0/3 [rapidfuzz]
   ----------------------------------------

In [2]:
import pandas as pd
import re
import time
import os
import hashlib
from fuzzywuzzy import process
from tqdm import tqdm  # For progress tracking


In [3]:
# ===========================
# Step 1: Load the Excel File
# ===========================


file_path = "ivntest.xlsx"  # Replace with your actual file name
output_file = "IVN_Dataset_Cleaned.xlsx"


def row_hash(row):
    return hashlib.md5(str(row.values).encode()).hexdigest()


# Load the input file and compute hashes for ALL rows
df_input = pd.read_excel(file_path, engine="openpyxl")
df_input['Row_Hash'] = df_input.apply(row_hash, axis=1)

# Load or initialize the output file
if os.path.exists(output_file):
    df = pd.read_excel(output_file, engine="openpyxl")
    print("Loaded existing cleaned data.")
    
    # Get set of existing hashes from the cleaned dataset
    if 'Row_Hash' not in df.columns:
        df['Row_Hash'] = None
    existing_hashes = set(df['Row_Hash'].dropna())
    
    # Identify ONLY new rows (those NOT already processed)
    new_row_mask = ~df_input['Row_Hash'].isin(existing_hashes)
    df_new = df_input[new_row_mask].copy()
    
    if not df_new.empty:
        print(f"Found {len(df_new)} new rows to process.")
        num_new = len(df_new)
    else:
        print("No new rows to process. Exiting.")
        df_new = pd.DataFrame()  # Empty dataframe to skip processing
        num_new = 0
else:
    print("Output file does not exist. Processing all rows as new.")
    df = pd.DataFrame()
    df_new = df_input.copy()
    num_new = len(df_new)


if 'Cleaned_Timestamp' not in df.columns:
    df['Cleaned_Timestamp'] = pd.NaT

if num_new == 0:
    print("Skipping processing - no new rows found.")
else:
    # Add Cleaned_Timestamp column to new rows if not present
    if 'Cleaned_Timestamp' not in df_new.columns:
        df_new['Cleaned_Timestamp'] = pd.NaT


    # Inspect the new rows
    print("\nNew Rows to Process:")
    print(df_new.head())


    # ===================================
    # Step 2: Define Cleaning Functions
    # ===================================


    def clean_text(text):
        """
        Cleans a given text string by:
        - Removing leading/trailing spaces
        - Standardizing spaces between words
        - Replacing en-dashes and em-dashes with hyphens
        - Ensuring consistent sentence spacing (one space after a period)
        - Standardizing quotation marks and apostrophes
        - Removing non-printable characters
        - Keeping original case (no lowercase conversion)
        """
        if not isinstance(text, str):  # Ensure input is a string
            return ""


        text = text.strip()  # Remove leading and trailing spaces
        text = re.sub(r"\s+", " ", text)  # Replace multiple spaces with a single space
        text = text.replace("–", "-").replace("—", "-")  # Replace en-dash and em-dash with hyphen
        text = text.replace(""", '"').replace(""", '"')  # Standardize double quotes
        text = text.replace("'", "'").replace("'", "'")  # Standardize single quotes
        text = re.sub(r"\.\s+", ". ", text)  # Ensure one space after periods
        text = re.sub(r"[^a-zA-Z0-9.,;:'\"!?()\-\s]", "", text)  # Remove special characters (preserving punctuation)
        text = text.strip()  # Ensure no trailing spaces remain
        return text


    # ==========================================
    # Step 3: Apply Cleaning to Relevant Columns
    # ==========================================


    # Define the columns that need cleaning
    columns_to_clean = ["Enabling Component", "Dependent Component"]


    for col in columns_to_clean:
        if col in df_new.columns:  # Ensure the column exists
            df_new[col] = df_new[col].apply(clean_text)  # Apply cleaning to new rows only


    # ===========================
    # Step 4: Deduplicate Entries (With Progress Tracking)
    # ===========================


    def deduplicate_column_partial(texts, unique_texts):
        """
        Uses fuzzy matching to deduplicate a list of texts using an existing unique_texts dict.
        Updates the unique_texts dict with new entries.
        """
        cleaned_column = []
        for text in tqdm(texts, desc="Deduplicating New Entries", unit="entry"):
            if not text.strip():
                cleaned_column.append(text)
                continue
            if text in unique_texts:
                cleaned_column.append(unique_texts[text])
            else:
                non_empty_keys = [k for k in unique_texts if k.strip()]
                if non_empty_keys:
                    result = process.extractOne(text, non_empty_keys, score_cutoff=90)
                    if result:
                        match, score = result
                        cleaned_column.append(unique_texts[match])
                    else:
                        unique_texts[text] = text
                        cleaned_column.append(text)
                else:
                    unique_texts[text] = text
                    cleaned_column.append(text)
        return cleaned_column


    # Apply deduplication to each relevant column
    for col in columns_to_clean:
        if col in df_new.columns:
            # Build unique_texts from already cleaned rows in the main dataset
            unique_texts = {}
            if not df.empty and 'Cleaned_Timestamp' in df.columns:
                cleaned_mask = df['Cleaned_Timestamp'].notna()
                for text in df.loc[cleaned_mask, col]:
                    if isinstance(text, str) and text.strip():
                        unique_texts[text] = text
            
            # Deduplicate the new rows
            new_texts = df_new[col].tolist()
            if new_texts:
                deduplicated_new = deduplicate_column_partial(new_texts, unique_texts)
                df_new[col] = deduplicated_new
    
    
    # =============================
    # Step 5: Ensure Consistent IDs
    # =============================


    if "Component ID" in df_new.columns:
        # Get max ID from the cleaned dataset
        if not df.empty and "Component ID" in df.columns:
            max_id = df["Component ID"].max() if not df["Component ID"].isna().all() else 0
        else:
            max_id = 0
        
        # Fill missing IDs in new rows with auto-generated numbers
        missing_id_mask = df_new["Component ID"].isna()
        num_missing = missing_id_mask.sum()
        if num_missing > 0:
            new_ids = range(int(max_id) + 1, int(max_id) + 1 + num_missing)
            df_new.loc[missing_id_mask, "Component ID"] = list(new_ids)
    
    
    # Set timestamp for new rows
    df_new['Cleaned_Timestamp'] = pd.Timestamp.now()


    # =========================
    # Step 6: Merge and Save Cleaned Data
    # =========================


    # Append new rows to the existing cleaned dataset
    if not df.empty:
        df = pd.concat([df, df_new], ignore_index=True)
    else:
        df = df_new


    # Save cleaned dataset to Excel file
    df.to_excel(output_file, index=False, engine="openpyxl")

    print(f"\n\nData cleaning complete! Added {len(df_new)} new rows.")
    print(f"Total rows in cleaned dataset: {len(df)}")
    print(f"Cleaned file saved as: {output_file}")

Loaded existing cleaned data.
Found 4088 new rows to process.

New Rows to Process:
                                     Enabling Source  \
0  AMS Strategic Plan 2020-2024 (latest version a...   
1                      USDA Strategic Plan 2022-2026   
2                                 USDA IT Directives   
3                                 USDA IT Directives   
4                                 USDA IT Directives   

                                  Enabling Component  \
0  AMS SP 20-24 4.3: Improve Access to Healthy, L...   
1  USDA SP 2022-2026 2.0: Ensure America's Agricu...   
2  USDA IT Dir 3300-001-J Emergency Communication...   
3      USDA IT DR 3520-002, Configuration Management   
4  USDA IT Dir 3145-001: Oversight and Management...   

                      Enabling Component Description  \
0  AMS promotes producer access to local and regi...   
1  A strong and prosperous agricultural sector is...   
2  This Departmental Regulation (DR) defines the ...   
3  This Memorandum

Deduplicating New Entries: 100%|██████████| 4088/4088 [00:10<00:00, 387.43entry/s]
C:\Users\Kristen.Boucher\AppData\Local\Temp\ipykernel_8228\2840932185.py:189: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, df_new], ignore_index=True)




Data cleaning complete! Added 4088 new rows.
Total rows in cleaned dataset: 8176
Cleaned file saved as: IVN_Dataset_Cleaned.xlsx
